In [1]:
import torch
import torch.nn as nn

from sklearn.preprocessing import  StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [2]:
data=load_breast_cancer()
X=data.data
y=data.target
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)
ss=StandardScaler()
X_train_scaled=ss.fit_transform(X_train)
X_test_scaled=ss.transform(X_test)



In [3]:
# ✅ Correct full code
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor  = torch.FloatTensor(X_test_scaled)
y_train_tensor = torch.LongTensor(y_train)    # ✅ LongTensor for labels
y_test_tensor  = torch.LongTensor(y_test)     # ✅ LongTensor for labels

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [4]:
class ANN(nn.Module):
    def __init__(self,input_size,hidden_size,num_classes):
        super(ANN,self).__init__()
        self.network=nn.Sequential(


            nn.Linear(input_size,hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),


            nn.Linear(hidden_size,hidden_size//2),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_size // 2, num_classes)



        )
    def forward(self,x):
        return self.network(x)
    


In [5]:
input_size=X_train.shape[1]
hidden_size=64
num_classes=2
model=ANN(input_size,hidden_size,num_classes)

crieterion=nn.CrossEntropyLoss()

optimizer=optim.Adam(model.parameters())





In [6]:
epochs=100
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = crieterion(outputs, yb)   # ✅ fixed spelling

        loss.backward()                 # ✅ no 's'
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")

RuntimeError: running_mean should contain 32 elements not 64

In [ ]:
model.eval()     # disables Dropout and BatchNorm (use learned stats)

with torch.no_grad():    # no gradient computation needed during eval
    outputs  = model(X_test)
    _, predicted = torch.max(outputs, 1)
    # torch.max returns (values, indices) — we want indices = predicted class

    correct  = (predicted == y_test).sum().item()
    accuracy = correct / len(y_test) * 100
    print(f"Test Accuracy: {accuracy:.2f}%")

TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

: 